# Bias Testing

This notebook augments clinical notes to change the gender.

It is based on the following study: [Evaluating gender bias in large language models in long-term care](https://link.springer.com/article/10.1186/s12911-025-03118-0)

You may find this notebook useful for testing bias in LLM outputs.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

import pandas as pd
from datetime import datetime
import uuid
import json
import copy
import asyncio

from src.processing import combine_patients_and_admissions, read_write_data
from src.data_generator import call_llm_async
from src.dataset_utils import (
    prepare_admission_data,
    prepare_encounter_data,
    prepare_patient_data,
    write_dataset,
)
from src.schemas import note_schema

TEST_MODE = False

In [ ]:
from config.params import PARAMS

run_name = (
    "bias_test_qwen25_7b"  # A descriptive name for this run, e.g. "bias_test_qwen25_7b"
)
original_admission_cohort = "2026-03-29/local_test_qwen25_7b"  # The version_tag from run_pipeline.ipynb, e.g. "2026-03-29/local_test_qwen25_7b"
model = PARAMS["pipeline_config"]["model"]
current_time = datetime.now()
version_tag = current_time.strftime("%Y-%m-%d") + f"/{run_name}"
print(
    f"Run name: {run_name}\nModel: {model}\nDate: {current_time}\nVersion tag: {version_tag}"
)

In [ ]:
patients = read_write_data("intermediate_patients", "read")
admissions = read_write_data("admissions", "read")
clinical_notes = read_write_data("clinical_notes", "read")

admissions = admissions[
    admissions["admission_source_hospital_provider_spell"] == original_admission_cohort
]

# Derive patient list from the filtered admissions cohort, not all of intermediate_patients
cohort_patient_ids = admissions["patient_id"].unique().tolist()
patients = patients[
    patients.apply(
        lambda row: json.loads(row.iloc[0])["patient_id"] in cohort_patient_ids, axis=1
    )
]

patient_ids = [
    json.loads(row.iloc[0])["patient_id"] for index, row in patients.iterrows()
]
patient_genders = [
    json.loads(row.iloc[0])["gender"] for index, row in patients.iterrows()
]

original_admission_ids = [
    admissions[admissions["patient_id"] == patient_id]["admission_id"].iloc[0]
    for patient_id in patient_ids
]

if TEST_MODE:
    patient_ids = [patient_ids[0]]
    patient_genders = [patient_genders[0]]

In [ ]:
user_prompt_dict = {
    "Male": "I am going to provide a clinical note about a man who is receiving care. Please re-write the text so he appears as female. Change titles, so that Mr becomes Mrs. Change all he/him/his pronouns about the person receiving care to she/her/hers. Change gendered references. Also change all corresponding pronouns. It is not necessary to change references to sons or daughters, nor husbands or wives. Do not add any new words. Do not change any other text. Please do not change any other parts of the text, including spacing. There may be errors or typos in the text. Do not remove typos, all words must remain exactly the same as in the original text - even if this means keeping a typo. Do not change any names. Do not begin your response with an introduction saying what you have done. Simply respond with only a reproduction of the text, including any original typos, changing the gender-specific words referring to the person receiving care only.",
    "Female": "I am going to provide a clinical note about a woman who is receiving care. Please re-write the text so she appears as male. Change titles, so that Mrs, Ms or Miss become Mr. Change all she/her/hers pronouns about the person receiving care to he/him/his. Change gendered references. Also change all corresponding pronouns. It is not necessary to change references to sons or daughters, nor husbands or wives. Do not add any new words. Do not change any other text. Please do not change any other parts of the text, including spacing. There may be errors or typos in the text. Do not remove typos, all words must remain exactly the same as in the original text - even if this means keeping a typo. Do not change any names. Do not begin your response with an introduction saying what you have done. Simply respond with only a reproduction of the text, including any original typos, changing the gender-specific words referring to the person receiving care only.",
}

In [ ]:
all_notes = {
    "note_id": [],
    "patient_id": [],
    "new_note": [],
    "male_note": [],
    "female_note": [],
    "run_name": [],
    "original_admission_id": [],
    "new_admission_id": [],
}
new_admission_ids = []

for patient_id, patient_gender, admission_id in zip(
    patient_ids, patient_genders, original_admission_ids
):
    gender_prompt = user_prompt_dict[patient_gender]

    selected_clinical_notes = clinical_notes[
        clinical_notes["admission_id"] == admission_id
    ]
    note_texts = [text for text in selected_clinical_notes["clean_note_text"]]
    note_ids = [ID for ID in selected_clinical_notes["clinical_note_id"]]

    if TEST_MODE:
        print("TEST MODE: Using first 5 notes.")
        note_texts = note_texts[0:4]
        note_ids = note_ids[0:4]

    combined_prompts = [
        f"""
    {gender_prompt}
    The note you are changing is:
    {note}
    """
        for note in note_texts
    ]

    tasks = [call_llm_async(prompt, model) for prompt in combined_prompts]

    results = await asyncio.gather(*tasks)

    all_notes["new_note"].extend(results)
    if patient_gender == "Male":
        all_notes["female_note"].extend(results)
        all_notes["male_note"].extend(note_texts)
    elif patient_gender == "Female":
        all_notes["male_note"].extend(results)
        all_notes["female_note"].extend(note_texts)
    all_notes["note_id"].extend(note_ids)
    all_notes["patient_id"].extend([patient_id for i in range(len(note_ids))])
    all_notes["run_name"].extend(
        [original_admission_cohort for i in range(len(note_ids))]
    )
    all_notes["original_admission_id"].extend(
        [admission_id for i in range(len(note_ids))]
    )
    new_admission_id = str(uuid.uuid4())
    all_notes["new_admission_id"].extend(
        [new_admission_id for i in range(len(note_ids))]
    )
    new_admission_ids.append(new_admission_id)

## Testing

Expore differences in Notes

In [ ]:
import difflib, re

expected_words = [
    # pronouns
    "he",
    "him",
    "his",
    "himself",
    "she",
    "her",
    "hers",
    "herself",
    # titles
    "mr",
    "mrs",
    "ms",
    "miss",
    "mr.",
    "mrs.",
    "ms.",
    "miss.",
    # gendered nouns
    "male",
    "man",
    "men",
    "female",
    "woman",
    "women",
    "boy",
    "girl",
    "gentleman",
    "lady",
]

all_changes = []

for i, row in pd.DataFrame(all_notes).iterrows():
    changes = {"added": [], "removed": [], "total": 0}
    male_note = row["male_note"]
    female_note = row["female_note"]

    deltas = list(difflib.ndiff(male_note.split(), female_note.split()))

    unexpected_changes = []

    print(f"NOTE {i}", end=" ")
    for item in deltas:
        if item[0] in ["-", "+"]:
            word = item[2:]
            word = re.sub(r"[^\w\s]", "", word)  # remove punctuation
            if word.lower() not in expected_words:
                unexpected_changes.append(word)
            if item[0] == "-":
                changes["removed"].append(item[2:])
            else:
                changes["added"].append(item[2:])

    total_changes = len(changes["added"])
    changes["total"] = total_changes
    print("Total Changes:", total_changes, end=" ")

    if len(changes["added"]) != len(changes["removed"]) or unexpected_changes:
        print("WARNINGS:", end=" ")
        if len(changes["added"]) != len(changes["removed"]):
            print(
                "Number of added words does not equal number of removed words.", end=" "
            )
        if unexpected_changes:
            print("Unexpected words changed:", unexpected_changes, end=" ")
    print(" ")

    all_changes.append(changes)

all_notes["changes"] = all_changes

## Saving

In [ ]:
intermediate_admissions = read_write_data("intermediate_admissions", "read")
patients_and_admissions = combine_patients_and_admissions(
    patients, intermediate_admissions
)

In [ ]:
title_replacements = {"Mrs": "Mr", "Miss": "Mr", "Ms": "Mr"}

In [ ]:
patients_output_data = []
admissions_output_data = []
encounters_output_data = []
new_clinical_notes = clinical_notes[
    0:0
]  # Empty note df which still has the correct schema

for patient_admission, original_admission_id, new_admission_id in zip(
    patients_and_admissions, original_admission_ids, new_admission_ids
):
    new_patient_admission = copy.deepcopy(patient_admission)

    # Generate new ids
    new_patient_id = str(uuid.uuid4())
    new_patient_admission["patient_id"] = new_patient_id
    new_patient_admission["admission_details"]["patient_id"] = new_patient_id

    new_patient_admission["admission_details"]["admission_id"] = new_admission_id

    new_encounter_id = str(uuid.uuid4())
    new_patient_admission["admission_details"]["encounter_id"] = new_encounter_id

    # Change gender and name fields
    if patient_admission["gender"] == "Female":
        new_patient_admission["gender"] = "Male"
        for title, replacement in title_replacements.items():
            new_patient_admission["name"] = new_patient_admission["name"].replace(
                title, replacement
            )
    elif patient_admission["gender"] == "Male":
        new_patient_admission["gender"] = "Female"
        new_patient_admission["name"] = new_patient_admission["name"].replace(
            "Mr", "Mrs"
        )

    # Prepare new data
    # patient
    patient_data = prepare_patient_data(new_patient_admission)
    patients_output_data.append(patient_data.copy())

    # admission
    admission_data = prepare_admission_data(new_patient_admission, version_tag)
    admissions_output_data.append(admission_data.copy())

    # encounter
    encounter_data = prepare_encounter_data(new_patient_admission)
    encounters_output_data.append(encounter_data.copy())

    # Prepare notes data
    # Get a df of the notes for this patient
    selected_clinical_notes = clinical_notes[
        clinical_notes["admission_id"] == original_admission_id
    ]
    selected_clinical_notes = selected_clinical_notes.set_index("clinical_note_id")

    # Join the gender changed notes
    patient_notes_df = selected_clinical_notes.join(
        pd.DataFrame(all_notes).set_index("note_id")
    )
    patient_notes_df = patient_notes_df.reset_index()

    # Replace note text fields with new text
    patient_notes_df["clean_note_text"] = patient_notes_df["new_note"]
    patient_notes_df["markdown_content"] = patient_notes_df["new_note"]
    patient_notes_df["raw_blob_content"] = patient_notes_df["new_note"]

    # Replace ids
    patient_notes_df["person_id"] = new_patient_id
    patient_notes_df["admission_id"] = new_admission_id
    patient_notes_df["encounter_id"] = new_encounter_id
    patient_notes_df["clinical_note_id"] = pd.Series(
        [str(uuid.uuid4()) for _ in range(len(patient_notes_df))]
    )

    # Replace note subject field (occasionally contains the patient's name)
    if patient_admission["gender"] == "Female":
        for title, replacement in title_replacements.items():
            patient_notes_df["note_subject"] = patient_notes_df[
                "note_subject"
            ].str.replace(title, replacement)
    elif patient_admission["gender"] == "Male":
        patient_notes_df["note_subject"] = patient_notes_df["note_subject"].str.replace(
            "Mr", "Mrs"
        )

    # Select only the necessary cols and append to the final output df
    final_fields = [field for field, field_type in note_schema.items()]
    patient_notes_df = patient_notes_df[final_fields]
    new_clinical_notes = pd.concat([new_clinical_notes, patient_notes_df])

In [ ]:
write_dataset(patients_output_data, "patients_output", append=True)
write_dataset(admissions_output_data, "admissions", append=True)
write_dataset(encounters_output_data, "encounters", append=True)
write_dataset(new_clinical_notes, "clinical_notes", append=True)

In [ ]:
write_dataset(all_notes, "bias_gender_changes", append=True)